In [21]:
# import
import requests
from bs4 import BeautifulSoup
import time
import csv
import os
from datetime import datetime, timedelta
from urllib.parse import urlparse, parse_qs

In [10]:
posts = get_post_list("aion2", 1)
print("게시글 개수:", len(posts))

for p in posts[:5]:
    print(p)

게시글 개수: 0


In [11]:
res = requests.get("https://gall.dcinside.com/board/lists?id=aion2", headers=HEADERS)
print(res.status_code)
print(res.text[:500])

200
<script type="text/javascript">location.replace("https://gall.dcinside.com/mgallery/board/lists?id=aion2");</script>


In [22]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 저장 폴더
OUTPUT_FOLDER = "dcinside_out"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# 아이온2 날짜 필터 설정
TARGET_DATE = datetime(2025, 11, 29)
START_DATE_AION = TARGET_DATE - timedelta(days=14)
END_DATE_AION = TARGET_DATE + timedelta(days=14)

def parse_date(date_str):
    try:
        return datetime.strptime(date_str, "%Y.%m.%d %H:%M")
    except:
        return None

def get_post_links(gallery_id, page=1, date_filter=False):
    url = f"https://gall.dcinside.com/mgallery/board/lists/?id={gallery_id}&page={page}"
    res = requests.get(url, headers=HEADERS)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")
    
    links = []
    for tr in soup.select("tr.ub-content"):
        a_tag = tr.select_one("td.gall_tit > a")
        date_tag = tr.select_one("span.gall_date")
        if not a_tag:
            continue
        href = a_tag.get("href", "")
        # 광고/JS 링크 제외
        if "javascript:" in href or "/mgallery/board/view/" not in href:
            continue
        # 날짜 필터
        if date_filter and date_tag:
            post_date = parse_date(date_tag.get_text(strip=True))
            if not post_date or not (START_DATE_AION <= post_date <= END_DATE_AION):
                continue
        full_link = "https://gall.dcinside.com" + href
        links.append(full_link)
    return links

def get_post_detail(post_url):
    res = requests.get(post_url, headers=HEADERS)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")
    
    # 제목
    title_tag = soup.select_one("div.title_subject > span")
    title = title_tag.get_text(strip=True) if title_tag else ""
    
    # 작성일
    date_tag = soup.select_one("span.gall_date")
    created_dt_str = date_tag.get_text(strip=True) if date_tag else ""
    
    # 본문
    content_tag = soup.select_one("div.writing_view_box")
    selftext = content_tag.get_text(strip=True) if content_tag else ""
    
    # 조회수/좋아요
    view_tag = soup.select_one("span.view_count")
    view_count = int(view_tag.get_text(strip=True)) if view_tag and view_tag.get_text(strip=True).isdigit() else 0
    like_tag = soup.select_one("span.btn_recommend em")
    like_count = int(like_tag.get_text(strip=True)) if like_tag and like_tag.get_text(strip=True).isdigit() else 0
    
    # 게시글 번호/갤러리 ID
    parsed = urlparse(post_url)
    post_no = parse_qs(parsed.query).get("no", [None])[0]
    gallery_id = parse_qs(parsed.query).get("id", [None])[0]
    
    # 댓글 수집
    comments = []
    if post_no and gallery_id:
        comment_url = f"https://gall.dcinside.com/board/comment_json/?id={gallery_id}&no={post_no}&page=1&listnum=1000"
        c_res = requests.get(comment_url, headers=HEADERS)
        if c_res.status_code == 200:
            try:
                c_json = c_res.json()
                for item in c_json.get("data", []):
                    comments.append(item.get("content", "").replace("\n", " "))
            except Exception:
                pass
    
    return {
        "title": title,
        "created_dt": created_dt_str,
        "selftext": selftext,
        "view_count": view_count,
        "like_count": like_count,
        "comments": comments,
        "link": post_url
    }

def crawl_gallery_csv(gallery_id, max_pages=50, delay=0.5, date_filter=False):
    output_file = os.path.join(OUTPUT_FOLDER, f"{gallery_id}_posts.csv")
    all_posts = []
    post_counter = 0
    
    for page in range(1, max_pages+1):
        post_links = get_post_links(gallery_id, page, date_filter=date_filter)
        if not post_links:
            print(f"[{gallery_id}] 페이지 {page}/{max_pages} → 게시글 없음 → 종료")
            break
        for link in post_links:
            try:
                post_data = get_post_detail(link)
                all_posts.append(post_data)
                post_counter += 1
                print(f"[{gallery_id}] 페이지 {page}/{max_pages}, 총 게시글: {post_counter}, 댓글: {len(post_data['comments'])}, 진행률: {(page/max_pages)*100:.1f}%")
            except Exception as e:
                print(f"[{gallery_id}] 오류 발생:", e)
            time.sleep(delay)
    
    # CSV 저장
    keys = ["title", "created_dt", "selftext", "view_count", "like_count", "comments", "link"]
    with open(output_file, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        for post in all_posts:
            post_copy = post.copy()
            post_copy["comments"] = " | ".join(post_copy["comments"])
            writer.writerow(post_copy)
    
    print(f"\n[{gallery_id}] CSV 저장 완료: {output_file}, 총 게시글 수: {len(all_posts)}")

In [ ]:
# 실행
crawl_gallery_csv("aion2", max_pages=50, delay=0.5, date_filter=True)
crawl_gallery_csv("lostarkm", max_pages=50, delay=0.5, date_filter=False)

[aion2] 페이지 1/50, 총 게시글: 1, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 2, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 3, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 4, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 5, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 6, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 7, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 8, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 9, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 10, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 11, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 12, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 13, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 14, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 15, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 16, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 17, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 18, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 19, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 20, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 21, 댓글: 0, 진행률: 2.0%
[aion2] 페이지 1/50, 총 게시글: 22, 댓글: 0, 진행률: 2.